# 07 — Querying the graph

Companion to **[Chapter 13](../13-querying-the-graph.md)**.

You built a knowledge graph over twelve chapters. Now you ask it questions.
Run the cells in order; each one maps to a numbered section of the chapter.

## Setup — section 13.2

One client, one set of view IDs, reused by every cell below.

In [ ]:
# ---------------------------------------------------------------- setup ----
import os
from pathlib import Path

from cognite.client import CogniteClient, global_config
global_config.disable_pypi_version_check = True
from cognite.client.config import ClientConfig
from cognite.client.credentials import OAuthClientCredentials, OAuthInteractive

# Find the repo root by its markers, so this cell works wherever Jupyter started.
HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents]
             if (p / "pyproject.toml").exists() and (p / "training").exists()), HERE)

env_path = ROOT / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#") or "=" not in s:
            continue
        k, v = s.split("=", 1)
        if " #" in v and not v.startswith(('"', "'")):
            v = v.split(" #", 1)[0].rstrip()
        os.environ.setdefault(k, v)      # a real environment variable always wins

missing = [k for k in ("CDF_PROJECT", "CDF_CLUSTER", "IDP_CLIENT_ID")
           if not os.environ.get(k)]
assert not missing, f"Missing {missing}. Copy .env.example to {env_path} and fill it in."


def cdf_client(name: str) -> CogniteClient:
    """Build the client EXPLICITLY.

    `CogniteClient()` with no arguments does not read your .env. The SDK removed
    implicit construction in v8 and raises:
        ValueError: No ClientConfig has been provided
    The branch below is the two-identity rule from Chapter 02, in code.
    """
    base_url = os.environ.get("CDF_URL") or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
    scopes = [s for s in os.environ.get("IDP_SCOPES", f"{base_url}/.default").split(",") if s]

    if os.environ.get("LOGIN_FLOW", "interactive").lower() == "interactive":
        creds = OAuthInteractive(              # you, in a browser -- needs
            authority_url=os.environ["IDP_AUTHORITY_URL"],   # localhost:53000
            client_id=os.environ["IDP_CLIENT_ID"],           # as a redirect URI
            scopes=scopes)
    else:
        creds = OAuthClientCredentials(        # unattended: a service principal
            token_url=os.environ["IDP_TOKEN_URL"],
            client_id=os.environ["IDP_CLIENT_ID"],
            client_secret=os.environ["IDP_CLIENT_SECRET"],
            scopes=scopes)

    return CogniteClient(ClientConfig(
        client_name=name, project=os.environ["CDF_PROJECT"],
        base_url=base_url, credentials=creds))


YOURNAME = os.environ.get("PARTICIPANT", "YOURNAME")   # [CHANGE] if not in .env
client   = cdf_client(f"dm-handson-{YOURNAME}-query")

space       = f"isp_{YOURNAME}_TRN"
schema_edm  = f"ssp_{YOURNAME}_TrainingCore_edm"
schema_sdm  = f"ssp_{YOURNAME}_MaintenanceInsight_sdm"
raw_db      = f"rwd_{YOURNAME}_Training_TRN"
model_version = "v1.0.0"


# --- identifiers every chapter uses ---------------------------------------
from cognite.client.data_classes.data_modeling import ViewId
from cognite.client.data_classes import filters as flt
from cognite.client.data_classes.data_modeling.query import (
    Query, QuerySync, NodeResultSetExpression, EdgeResultSetExpression,
    Select, SourceSelector)
from cognite.client.data_classes.data_modeling import (
    NodeId, EdgeId, NodeApply, EdgeApply, NodeOrEdgeData, DirectRelationReference)
from cognite.client.data_classes.raw import Row

from cognite.client.data_classes.aggregations import Count, Avg, Max

INSTANCE_SPACE = space
EDM_SPACE      = schema_edm
SDM_SPACE      = schema_sdm
RAW_DB         = raw_db
MODEL_VERSION  = model_version

ASSET      = ViewId("cdf_cdm", "CogniteAsset",     "v1")
EQUIPMENT  = ViewId("cdf_cdm", "CogniteEquipment", "v1")
ACTIVITY   = ViewId("cdf_cdm", "CogniteActivity",  "v1")
TIMESERIES = ViewId("cdf_cdm", "CogniteTimeSeries","v1")
FILE       = ViewId("cdf_cdm", "CogniteFile",      "v1")
WORKORDER  = ViewId(EDM_SPACE, "WorkOrder",              MODEL_VERSION)
EHP        = ViewId(SDM_SPACE, "EquipmentHealthProfile", MODEL_VERSION)

print("connected:", client.config.project, "| space:", space)

## section 13.2 — Count what you built

The cheapest possible question. Note what this does *not* do: it never transfers the nodes.

In [ ]:
res = client.data_modeling.instances.aggregate(
    view=ASSET,
    aggregates=Count("externalId"),
    filter=flt.SpaceFilter(INSTANCE_SPACE, "node"),
)
print(res)          # expect 8

## section 13.4 — Find the failing pump

Your first `/query`: a single result set, no traversal.

In [ ]:
q = Query(
    with_={
        "pump": NodeResultSetExpression(
            filter=flt.And(
                flt.SpaceFilter(INSTANCE_SPACE, "node"),
                flt.Equals(["node", "externalId"], "21-PA-2001A"),
            ),
            limit=1,
        )
    },
    select={"pump": Select([SourceSelector(ASSET, ["name", "description", "tags"])])},
)

result = client.data_modeling.instances.query(q)
for node in result["pump"]:
    print(node.external_id, "|", node.properties[ASSET])

## section 13.5 — Which work orders touch the pump?

A list of direct relations cannot be walked backwards. Meet the error, then use the
filter that does work.

In [ ]:
# Your first instinct is to traverse inwards. It does not work -- run it and read
# the error, because the reason matters more than the workaround.
WORKORDER = ViewId(EDM_SPACE, "WorkOrder", MODEL_VERSION)
try:
    client.data_modeling.instances.query(Query(
        with_={
            "pump": NodeResultSetExpression(
                filter=flt.Equals(["node", "externalId"], "21-PA-2001A"), limit=1),
            "orders": NodeResultSetExpression(
                from_="pump", through=WORKORDER.as_property_ref("assets"),
                direction="inwards", limit=100),
        },
        select={"orders": Select([SourceSelector(WORKORDER, ["workOrderNumber"])])}))
except Exception as e:
    print("EXPECTED FAILURE ->", str(e).split("|")[0].strip())

# `assets` is a LIST of direct relations. DMS keeps no reverse index for list
# membership, so it cannot be walked backwards. Filter instead of traversing.
PUMP = {"space": INSTANCE_SPACE, "externalId": "21-PA-2001A"}

orders = client.data_modeling.instances.list(
    sources=WORKORDER, space=INSTANCE_SPACE, limit=-1,
    filter=flt.ContainsAny(WORKORDER.as_property_ref("assets"), [PUMP]))

for n in orders:
    p = n.properties[WORKORDER]
    print(f"   {p['workOrderNumber']:<9} {str(p['status']):<12} "
          f"{p.get('actualCost')} {p.get('currency')}")
# expect exactly WO-1001 -- the other two point at the separator

## section 13.5 (cont.) -- walking a *single* direct relation backwards

`WorkOrder.assets` is a list, so it is a dead end inwards. `EquipmentHealthProfile.asset`
is a single direct relation, and that one reverses. This is why Chapter 03 declares
`healthProfile` on the `Asset` view.

In [ ]:
# Identical query shape to the one that just failed -- one property, not a list.
q = Query(
    with_={
        "pump": NodeResultSetExpression(
            filter=flt.Equals(["node", "externalId"], "21-PA-2001A"), limit=1),
        "profile": NodeResultSetExpression(
            from_="pump",
            through=EHP.as_property_ref("asset"),   # the FORWARD property
            direction="inwards", limit=10),
    },
    select={"profile": Select([SourceSelector(EHP, ["ratedPowerKw", "sealType"])])},
)
res = client.data_modeling.instances.query(q)
for n in res["profile"]:
    print(n.external_id, n.properties[EHP])
print(f"\n{len(res['profile'])} profile(s) -- no 'Cannot traverse lists' error, "
      "because `asset` is singular.")

# `through` names the FORWARD property (EquipmentHealthProfile.asset), not the
# `healthProfile` property declared on Asset. /query walks the relation directly.
# The declaration does not enable the traversal -- it PUBLISHES it, so GraphQL,
# Fusion, Canvas and Atlas AI agents can find it without hand-written queries.

## section 13.6 — The operations, and a surprise

Querying a CDM view returns everything that implements it.

In [ ]:
# Same shape of question for the operations -- with a surprise.
ACTIVITY = ViewId("cdf_cdm", "CogniteActivity", "v1")

acts_on_pump = client.data_modeling.instances.list(
    sources=ACTIVITY, space=INSTANCE_SPACE, limit=-1,
    filter=flt.ContainsAny(ACTIVITY.as_property_ref("assets"), [PUMP]))
print("activities on the pump:", sorted(n.external_id for n in acts_on_pump))
# WO-1001 is in there -- a WORK ORDER. WorkOrder implements CogniteActivity,
# so querying the parent view returns both concepts.

wo_ids = {n.external_id for n in client.data_modeling.instances.list(
    sources=WORKORDER, space=INSTANCE_SPACE, limit=-1)}
all_acts = client.data_modeling.instances.list(
    sources=ACTIVITY, space=INSTANCE_SPACE, limit=-1)
ops = [a for a in all_acts if a.external_id not in wo_ids]

print(f"activities: {len(all_acts)} | work orders: {len(wo_ids)} | operations: {len(ops)}")
# expect 9 | 3 | 6   <- hold on to that 6, chapter 14 opens with it

## section 13.7 — The `hasData` filter you did not write

`inspect()` is ground truth: it reports which *containers* a node has data in,
independently of any view. If a node shows in `inspect` but not through the view,
you have met the implicit `hasData` filter.

In [ ]:
from cognite.client.data_classes.data_modeling.instances import InvolvedContainers

through_view = client.data_modeling.instances.list(
    sources=EHP, space=INSTANCE_SPACE, limit=-1)
print("through the view :", len(through_view))

# inspect() is ground truth: which CONTAINERS does the node actually populate?
# It needs to be told what to report -- involved_containers or involved_views.
registry = client.data_modeling.instances.inspect(
    nodes=(INSTANCE_SPACE, "ehp_21-PA-2001A"),
    involved_containers=InvolvedContainers())
print("in the registry  :", registry)

## section 13.8 — Aggregate: the shape of the backlog

In [ ]:
# group_by does NOT accept enum properties, and `status` is an enum.
for bucket in client.data_modeling.instances.aggregate(
    view=WORKORDER, aggregates=Count("externalId"), group_by="orderType",
    filter=flt.SpaceFilter(INSTANCE_SPACE, "node"),
):
    print(bucket)          # PM01 -> 2, PM02 -> 1

try:
    client.data_modeling.instances.aggregate(
        view=WORKORDER, aggregates=Count("externalId"), group_by="status",
        filter=flt.SpaceFilter(INSTANCE_SPACE, "node"))
except Exception as e:
    print("EXPECTED FAILURE ->", str(e).split("|")[0].strip())

# To count one enum value, filter instead of grouping.
open_wo = client.data_modeling.instances.list(
    sources=WORKORDER, space=INSTANCE_SPACE, limit=-1,
    filter=flt.Equals(WORKORDER.as_property_ref("status"), "OPEN"))
print("open:", [n.external_id for n in open_wo])

print(client.data_modeling.instances.aggregate(
    view=WORKORDER, aggregates=[Avg("actualCost"), Max("actualCost")],
    filter=flt.SpaceFilter(INSTANCE_SPACE, "node")))
# WO-1002 has no actualCost. A null is not a zero -- aggregates skip it.

## section 13.9 — Search: when a human typed the word

In [ ]:
for h in client.data_modeling.instances.search(
    view=WORKORDER, query="seal", properties=["name", "description"],
    filter=flt.SpaceFilter(INSTANCE_SPACE, "node"), limit=10,
):
    print(h.external_id, "|", h.properties[WORKORDER]["workOrderNumber"])

## section 13.10 — Sync: read once, then only the changes

The second call returns 0. Change a work order in Fusion and run it again —
exactly one comes back.

In [ ]:
sq = QuerySync(
    with_={"orders": NodeResultSetExpression(
        # SpaceFilter alone syncs EVERY node in the space. HasData narrows it to
        # instances that actually carry WorkOrder data.
        filter=flt.And(flt.SpaceFilter(INSTANCE_SPACE, "node"),
                       flt.HasData(views=[WORKORDER])),
        limit=100)},
    select={"orders": Select([SourceSelector(WORKORDER, ["workOrderNumber", "status"])])},
)

first = client.data_modeling.instances.sync(sq)
print("initial:", len(first["orders"]), "orders")

sq.cursors = first.cursors
again = client.data_modeling.instances.sync(sq)
print("since then:", len(again["orders"]), "changed")

## Gate

- [ ] 8 assets counted
- [ ] Traversal returns exactly `WO-1001`, and you can say why not the other two
- [ ] Two-hop query returns operations, including `WO-9999-0010`
- [ ] `inspect()` shows which containers your EHP node populates
- [ ] Status aggregate gives three buckets of 1
- [ ] Second `sync` returns 0

→ **[Chapter 14 — Debugging broken links](../14-debugging-broken-links.md)**